# Lesson 21 | What changes when the network becomes larger?

A design that looks balanced on a tiny graph can become limited by memory, queues, or a small set of high-traffic neurons at larger scale.

Today asks one question:

> **How do we identify which stage is limiting the workload at a particular scale?**

Primary new concept: **bottleneck as the stage with the highest utilization relative to capacity.**

## 1. Concept ledger

**Already known:** latency, throughput, bandwidth, FIFO backpressure, DDR, sparse fan-out.

**New today:** **utilization** and a scale-dependent **bottleneck**.

**Preview only:** measured 10K/50K MaleCNS runs, bank conflicts, telemetry, and optimization such as caching or multiple synapse engines.

## 2. Bottleneck is a relationship, not a permanent label

For one stage:

[
utilization = rac{demand}{capacity}
]

A stage near or above 1.0 has little or no headroom.

The bottleneck can move when the graph, event rate, memory pattern, or architecture changes. “DDR is always the bottleneck” is not a specification.

## 3. Pipeline picture

```mermaid
flowchart LR
  A["spike source"] --> B["FIFO"]
  B --> C["synapse lookup"]
  C --> D["DDR / storage"]
  D --> E["target update"]
```

Every stage can have a different capacity and demand.

## 4. Run: synthetic scale study

These are **teaching numbers**, not measured FPGA results. They exist only to practice the analysis method.

In [ ]:
capacities = {
    "fifo": 10.0,
    "lookup": 8.0,
    "memory": 6.0,
    "update": 9.0,
}

cases = {
    "small": {"fifo": 2.0, "lookup": 3.6, "memory": 1.8, "update": 2.2},
    "medium": {"fifo": 5.0, "lookup": 5.5, "memory": 5.7, "update": 4.8},
    "large": {"fifo": 9.5, "lookup": 6.5, "memory": 5.4, "update": 6.8},
}

for name, demand in cases.items():
    utilization = {
        stage: demand[stage] / capacities[stage]
        for stage in capacities
    }
    bottleneck = max(utilization, key=utilization.get)
    print(name, "bottleneck:", bottleneck,
          "utilization:", round(utilization[bottleneck], 2))

## 5. Observe

The highest utilization is lookup in the small case, memory in the medium case, and FIFO in the large case.

That is the point of the lesson: a bottleneck is determined by **demand relative to capacity** and can move as workload scale and traffic distribution change. This toy diagnosis says where to investigate first; it does not automatically prove the deeper cause.

## 6. Hotspots are uneven work

Two networks can have the same total number of edges but very different traffic distributions. A few high-fanout or high-rate sources can create queue pressure or bank conflicts.

So “network size” is not a complete performance description. Distribution matters.

## 7. Measure before optimizing

RMD-022 intentionally comes after the correctness baseline.

Caching, banking, lazy updates, or more engines are not automatically improvements. First establish:

1. a correct baseline;
2. telemetry and workload definition;
3. the observed limiting resource;
4. a before/after benchmark with the same workload.

## 8. Try It

In the medium case, increase only memory capacity. Predict where the bottleneck moves.

Then in the large case, increase only FIFO capacity. Which stage now has the highest utilization?

## 9. Exercise

[Lesson 21 exercise: identify the highest-utilization stage](../../exercises/en/21_scaling_bottlenecks.ipynb)

## 10. AI Task

Give an AI a utilization table and ask for three possible causes of the hottest stage. Require it to label them as hypotheses, not measurements.

## 11. Human Check

Explain why the bottleneck can move with scale. Why is “largest raw workload” not necessarily the same as “highest utilization”? What evidence is required before choosing an optimization?

## 12. Engineering Handoff

Maps to `RMD-019~022`, `MOD-014 telemetry`, and performance metrics `P-001~P-008`. Formal claims require measured reports on a defined workload; the numbers in this lesson are only teaching fixtures.

## 13. Project Trace

- Lesson: `LSN-021`
- Mapping: `RMD-019 / RMD-020 / RMD-021 / RMD-022`
- Requirement paths: `TRACE-P-001`, `TRACE-F-001`
- Correctness oracle before optimization: `T-016`
- Performance metrics: `P-001~P-008`

## 14. Exit Ticket

Given demand and capacity for several stages, you can compute utilization, identify the current bottleneck candidate, and explain why that diagnosis may change at another scale.